# 05 — End-to-End Generation

Runs the real `answer_question()` from `retrieval.py` against the real corpus and the real Ollama model configured in `.env` — the exact function `app.py` calls for every live query.

In [1]:
import sys, os
sys.path.insert(0, "..")
os.chdir("..")

from langchain_community.vectorstores import FAISS

from core import FAISS_DIR, get_embeddings, get_llm, load_chunks
from retrieval import answer_question, build_ensemble_retriever

chunks = load_chunks()
embeddings = get_embeddings()
vectorstore = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)
retriever = build_ensemble_retriever(vectorstore, chunks)
llm = get_llm()
print("Engine loaded:", len(chunks), "chunks indexed")


/var/folders/8m/z6hx0h5s18d_brpspkfpd2_w0000gn/T/ipykernel_10508/3101880599.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Engine loaded: 882 chunks indexed


In [2]:
question = "What is the formula for the break-even point, and what do the variables stand for?"
result = answer_question(question, "all", retriever, llm)

print("ANSWER:\n", result.answer)
print("\nGROUNDED:", result.grounded)
print("\nSOURCES:")
for s in result.sources:
    print(f"  [{s.rank}] {s.file} -- {s.location}")
    print("      ", s.snippet[:120])


ANSWER:
 The formula for the break-even point is calculated as follows:

BEP = FC / (R - VC)

Where:

- BEP = Break-Even Point
- FC = Fixed Cost
- R = Revenue per unit
- VC = Variable Cost per unit

This formula indicates that the break-even point is the level of sales at which the total revenue equals the total costs, resulting in neither profit nor loss.

GROUNDED: True

SOURCES:
  [1] idm mids 1.pdf -- page 1
       a
 
mathematical
 
relationship
 
between
 
independent
 
variables
 
and
 
a
 
dependent
 
continuous
 
variable.
 
○
 
  [2] Unit 3 Finance and Marketing for early entrepreneurs.pdf -- page 20
       20 
 The break -even point is the level of sales at which total revenue equals total costs, resulting in 
zero profit or
  [3] Unit 3 Finance and Marketing for early entrepreneurs.pdf -- page 19
       :  For companies with significant intangible assets, suc h as patents, 
trademarks, or brand value, specific methods lik


And the guardrail, on a real out-of-scope question:

In [3]:
result2 = answer_question("What is the capital of France?", "all", retriever, llm)
print("ANSWER:", result2.answer)
print("GROUNDED:", result2.grounded)
print("SOURCES:", result2.sources)


ANSWER: I don't have information on that in the indexed documents.
GROUNDED: False
SOURCES: []
